# Tuần 7–9 — Đánh giá tracker trên SoccerNet-Tracking + render demo
Add Input: Kaggle Dataset chứa `best.pt` từ notebook train.

In [ ]:
# Bật GPU (T4 x2 hoặc P100) + Internet trong Settings của notebook
!git clone https://github.com/thaisanh1602/football_cv_tracking.git
%cd football_cv_tracking
!pip install -q -r requirements.txt

In [ ]:
WEIGHTS = "/kaggle/input/<your-weights-dataset>/best.pt"  # sửa đường dẫn

## Tải SoccerNet-Tracking (split test)

In [ ]:
from SoccerNet.Downloader import SoccerNetDownloader
SoccerNetDownloader(LocalDirectory="/kaggle/working/SoccerNet").downloadDataTask(task="tracking", split=["test"])
!cd /kaggle/working/SoccerNet/tracking && unzip -q test.zip

## Ablation: ByteTrack vs BoT-SORT vs BoT-SORT + ReID trên cùng subset

In [ ]:
DATA = "/kaggle/working/SoccerNet/tracking/test"
TRACKERS = "configs/bytetrack.yaml configs/botsort.yaml configs/botsort_reid.yaml configs/ablation/bytetrack_newtrack08.yaml configs/botsort_tuned.yaml"
# detection được cache nên 1 lệnh chạy nhiều tracker chỉ tốn detect 1 lần
!python scripts/eval_soccernet.py --data-root {DATA} --weights {WEIGHTS} --limit 10 --out /kaggle/working/trackeval --trackers {TRACKERS}
!python scripts/analyze_idsw.py --data-root {DATA} --results /kaggle/working/trackeval/bytetrack/data /kaggle/working/trackeval/botsort_tuned/data

In [ ]:
import json, pandas as pd
from pathlib import Path

rows = {p.parent.name: {k: v for k, v in json.loads(p.read_text()).items() if k != "seqs"} for p in Path("/kaggle/working/trackeval").glob("*/summary.json")}
pd.DataFrame(rows).T.round(2)  # copy bảng này vào README

## Render demo video

In [ ]:
SEQ = f"{DATA}/SNMOT-116"  # chọn 1 sequence đẹp
!ffmpeg -y -loglevel error -framerate 25 -i {SEQ}/img1/%06d.jpg -c:v libx264 -pix_fmt yuv420p /kaggle/working/clip.mp4
!python scripts/infer_video.py --source /kaggle/working/clip.mp4 --weights {WEIGHTS} --sahi --team --output /kaggle/working/demo.mp4